#  Live RF Model Evalutaion Against Current Market Data

This notebook evaluates the Random Forest volatility model against the most recent market data availabe. 

There are 2 parts:
1. Compare archived live prediction runs across target dates.
2. Evaluate a completed 20-trading-day window where actual future volatility is now known.

## Important: predictions for today/tomorrow cannot be fully judged until 20 future trading days have passed.


# Imports

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Paths

In [ ]:
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

PREDICTIONS_PATH = PROJECT_ROOT / "predictions"
REPORTS_PATH = PROJECT_ROOT / "reports"

prediction_log_path = PREDICTIONS_PATH / "prediction_log.csv"
evaluation_path = REPORTS_PATH / "latest_20d_rf_evaluation.csv"
summary_path = REPORTS_PATH / "latest_20d_rf_evaluation.summary.csv"

prediction_log_path, evaluation_path, summary_path

(WindowsPath('c:/Users/sshab/Documents/coding_projects/INFO442-Group-Project/predictions/prediction_log.csv'),
 WindowsPath('c:/Users/sshab/Documents/coding_projects/INFO442-Group-Project/reports/latest_20d_rf_evaluation.csv'),
 WindowsPath('c:/Users/sshab/Documents/coding_projects/INFO442-Group-Project/reports/latest_20d_rf_evaluation.summary.csv'))

# Load Archived Live Predictions

The prediction log stores each daily prediction run in long format:
- 'Date' : target prediction Date
- 'ticker' : asset symbol
- 'predicted_future_volatility_20d' : model forecast for future 20-day volatility.

In [6]:
prediction_log = pd.read_csv(prediction_log_path, parse_dates=['Date'])

In [7]:
prediction_log.head()

,Date,ticker,predicted_future_volatility_20d
0,2026-08-05,AAPL,0.018918
1,2026-08-05,AGG,0.002464
2,2026-08-05,AMZN,0.018553
3,2026-08-05,CAT,0.017330
4,2026-08-05,GLD,0.014319


# Data Summary

In [10]:
prediction_log_summary = (
    prediction_log
    .groupby("Date")
    .agg(
        tickers=("ticker", "nunique"),
        mean_predicted_volatility=("predicted_future_volatility_20d", "mean"),
        min_predicted_volatility=("predicted_future_volatility_20d", "min"),
        max_predicted_volatility=("predicted_future_volatility_20d", "max"),
    )
    .reset_index()
)

In [13]:
prediction_log_summary

,Date,tickers,mean_predicted_volatility,min_predicted_volatility,max_predicted_volatility
0,2026-08-05,21,0.014474,0.002464,0.018918
1,2026-08-06,21,0.014178,0.002270,0.018328


# Compare Prediction Runs Across Days

This chart will show how the model's predicted volatility changed between archived target dates.

In [14]:
fig = px.line(
    prediction_log,
    x="Date",
    y="predicted_future_volatility_20d",
    color="ticker",
    markers=True,
    title="Predicted 20-Day Volatility by Ticker Across Live Runs",
)

fig.update_layout(
    xaxis_title="Prediction Target Date",
    yaxis_title="Predicted Future 20-Day Volatility",
    legend_title="Ticker",
)

fig.show()

In [15]:
latest_two_dates = sorted(prediction_log["Date"].unique())[-2:]

prediction_change = (
    prediction_log[prediction_log["Date"].isin(latest_two_dates)]
    .pivot(index="ticker", columns="Date", values="predicted_future_volatility_20d")
)

prediction_change["change"] = prediction_change.iloc[:, -1] - prediction_change.iloc[:, -2]
prediction_change["absolute_change"] = prediction_change["change"].abs()

prediction_change = prediction_change.sort_values("absolute_change", ascending=False)

prediction_change.head(10)

Date,2026-08-05 00:00:00,2026-08-06 00:00:00,change,absolute_change
ticker,,,,
GLD,0.014319,0.016161,0.001842,0.001842
VNQ,0.009939,0.008357,-0.001582,0.001582
PG,0.015553,0.014053,-0.001500,0.001500
AAPL,0.018918,0.017746,-0.001172,0.001172
JPM,0.016056,0.015071,-0.000985,0.000985
NEE,0.011503,0.012203,0.000700,0.000700
MSFT,0.018421,0.017866,-0.000556,0.000556
LMT,0.016416,0.015908,-0.000509,0.000509
SPY,0.011617,0.011183,-0.000434,0.000434


In [16]:
fig = px.bar(
    prediction_change.reset_index(),
    x="ticker",
    y="change",
    title="Change in Predicted Volatility Between Latest Two Runs",
)

fig.update_layout(
    xaxis_title="Ticker",
    yaxis_title="Prediction Change",
)

fig.show()

# Load Completed 20-Day Evaluation

This file evaluates a historical prediction date where the next 20 trading days have already happened.

That lets us compare:

predicted_future_volatility_20d VS actual_future_volatility_20d